# Capstone: Build a mini-GPT from scratch

_Capstones_

**Seven landmark papers, built in order, become one small character-level GPT that generates text.**

---

This notebook is a practice scaffold for the **Capstone: Build a mini-GPT from scratch** lesson. We assemble the model one small piece at a time, and after each piece we **stop to look at the data**.

The heart of the notebook is **attention built tensor by tensor** on a real context: we watch the embeddings, the query/key/value projections, the raw scores, the causal mask, the softmax weights, and the value mix appear one operation at a time — *then* package them into a reusable module. Around that we inspect the vocabulary, the parameter budget, a single training step, the loss curve, and the trained model's predictions.

Cells marked **🎛️ Play with it** are interactive sandboxes: in Colab they show sliders and dropdowns you can drag to see how each operation responds; run them, then *change the values* — the fastest way to build intuition. Run each cell top to bottom and read the note above it. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / matplotlib ship with Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## First, look at the data

A language model just predicts the **next character**, so our "data" is a string, a table mapping each character to an integer id, and batches of `(input, next-char)` pairs. We build and inspect all three before touching the model.

### Step 1 — The character corpus

We train on a tiny, repeated snippet so the run is fast and the model can actually learn the pattern. Working at the **character** level keeps the vocabulary tiny (just the distinct characters), so the whole pipeline stays small enough to read end to end.

In [ ]:
# A tiny char-level corpus: two short lines, repeated so there is enough to learn from.
text = ("to be, or not to be, that is the question. "
        "whether tis nobler in the mind to suffer. ") * 50

print("total characters :", len(text))
print("distinct characters:", len(set(text)))
print("first 90 characters:")
print(repr(text[:90]))

### Step 2 — The vocabulary (every character gets an id)

A neural network consumes numbers, not letters, so we assign each distinct character an integer id. `stoi` ("string-to-int") and `itos` ("int-to-string") are the two lookup directions. The vocabulary size `VOCAB` matters: a freshly initialised model guesses uniformly over `VOCAB` characters, so its loss should start near `ln(VOCAB)`.

The table lists every character with its id and how often it appears; the bar chart shows the same frequencies (spaces and common letters dominate).

In [ ]:
import collections

chars = sorted(set(text))
stoi  = {c: i for i, c in enumerate(chars)}      # char -> id
itos  = {i: c for c, i in stoi.items()}          # id   -> char
VOCAB = len(chars)
counts = collections.Counter(text)

vocab_df = pd.DataFrame({
    "id":    [stoi[c] for c in chars],
    "char":  [repr(c) for c in chars],
    "count": [counts[c] for c in chars],
})
print("VOCAB size:", VOCAB,
      " ->  random-init loss should start near ln(VOCAB) =", round(float(np.log(VOCAB)), 3))
display(vocab_df)

plt.figure(figsize=(9, 2.6))
plt.bar([repr(c) for c in chars], [counts[c] for c in chars], color="#4ea1ff")
plt.title("character frequencies in the corpus")
plt.ylabel("count")
plt.show()

### Step 3 — Tokenize: turn text into integer ids

"Tokenizing" here just means mapping each character through `stoi`. The little table shows one phrase encoded character by character; then we encode the **whole corpus** into one long 1-D tensor `data` — the stream the model slides a window over.

In [ ]:
import torch

def encode(s):
    return [stoi[c] for c in s]

phrase = "to be"
enc_df = pd.DataFrame({
    "position": range(len(phrase)),
    "char":     [repr(c) for c in phrase],
    "id":       encode(phrase),
})
display(enc_df)

data = torch.tensor(encode(text))
print("encoded corpus tensor:", tuple(data.shape), data.dtype)
print("first 20 ids:", data[:20].tolist())

**🎛️ Play with it — your own text through the tokenizer**

Tokenizing is just a dictionary lookup, but it is where "text" becomes "tensor". Type any string and watch it become ids; characters our tiny corpus never used have no id and are skipped. **Try:** your name, punctuation, digits. **Watch:** which characters survive. **Why it matters:** a model can only ever see tokens that are in its vocabulary.

In [ ]:
# In Colab a text box appears below — edit it and the encoding updates live.
def show_encoding(s="hello there"):
    known   = [c for c in s if c in stoi]
    unknown = sorted({c for c in s if c not in stoi})
    print("text    :", repr(s))
    print("encoded :", [stoi[c] for c in known])
    if unknown:
        print("skipped (not in vocab):", [repr(c) for c in unknown])
    display(pd.DataFrame({"char": [repr(c) for c in known],
                          "id":   [stoi[c] for c in known]}))

try:
    from ipywidgets import interact, Text
    interact(show_encoding, s=Text(value="hello there"))
except Exception:
    show_encoding()        # no ipywidgets -> just show the default

### Step 4 — Batches and the next-character target (shift by one)

For every position the **label is simply the next character**, so the target `y` is the input `x` shifted left by one. `get_batch` draws `B` random windows of length `SEQ`. The table aligns the first window's input characters with their targets — read it as "after `'t'` comes `'o'`, after `'o'` comes `' '`, …".

In [ ]:
SEQ, B = 32, 64                  # window length, batch size
torch.manual_seed(0)

def get_batch():
    ix = torch.randint(0, len(data) - SEQ - 1, (B,))
    x = torch.stack([data[i:i + SEQ]     for i in ix])      # the window
    y = torch.stack([data[i + 1:i + SEQ + 1] for i in ix])  # the SAME window shifted left by 1
    return x, y

xb, yb = get_batch()
print("x batch:", tuple(xb.shape), "  y batch:", tuple(yb.shape))

k = 12
shift_df = pd.DataFrame({
    "pos":            range(k),
    "input id":       xb[0, :k].tolist(),
    "input char":     [repr(itos[i]) for i in xb[0, :k].tolist()],
    "-> target char": [repr(itos[i]) for i in yb[0, :k].tolist()],
    "target id":      yb[0, :k].tolist(),
})
display(shift_df)   # each input char's label is just the NEXT char

## Reference implementation — PyTorch

The finished model is a **decoder-only Transformer** (the GPT family), and every piece traces to one landmark paper. Here is the build plan:

| Paper | Contributes | In our code |
|---|---|---|
| word2vec | token embedding table | `nn.Embedding(VOCAB, d_model)` |
| attention | scaled dot-product attention | `Q @ Kᵀ / √dₖ` then `softmax` |
| transformer | multi-head attention, positions, the block | `CausalSelfAttention`, `nn.Embedding(max_len, …)`, `Block` |
| layernorm | pre-norm normalization | `nn.LayerNorm` before each sublayer |
| resnet | residual connections | the two `x + sublayer(x)` lines |
| adamw | decoupled weight decay | `torch.optim.AdamW` |
| gpt | causal mask + next-token loss + sampling | `masked_fill(mask, -inf)`, `cross_entropy`, `generate` |

We will build the trickiest part — attention — **tensor by tensor on a concrete context** first, then package it into a class. Pick a short context to follow through every step:

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F

D_MODEL = 64                          # width of every token vector
ctx = "to be, or not"                 # the context we will follow through attention
ids = torch.tensor([encode(ctx)])     # (1, S) integer ids
S = ids.shape[1]
print("context:", repr(ctx), "  ids:", ids.tolist(), "  S =", S)

### Step 5 — Token embeddings (a vector per character)

The first thing GPT does is look up each character id in a learnable **embedding table** — a `(VOCAB, D_MODEL)` matrix — giving every character a `D_MODEL`-dimensional vector (word2vec's idea). The heatmap shows the character-vectors for our context, each a row of 64 (random, untrained) numbers.

In [ ]:
torch.manual_seed(0)
tok_emb = nn.Embedding(VOCAB, D_MODEL)      # the lookup table (paper-word2vec)
tok = tok_emb(ids)                          # (1, S) ids -> (1, S, D_MODEL) vectors
print("ids   :", tuple(ids.shape), " ->  token vectors:", tuple(tok.shape))

plt.figure(figsize=(8, 3))
plt.imshow(tok[0].detach(), aspect="auto", cmap="coolwarm")
plt.yticks(range(S), list(ctx)); plt.xlabel("embedding dimension (0..63)")
plt.title("token embedding vectors — one row per character")
plt.colorbar(label="value"); plt.show()

### Step 6 — Positional embeddings (so order matters)

Attention by itself is order-blind, so we add a second learnable table indexed by **position** `0, 1, 2, …`. Adding the positional vector to the token vector gives the input `x` that attention actually sees — same shape, but now `"to"` at the start differs from `"to"` later on.

In [ ]:
pos_emb = nn.Embedding(SEQ, D_MODEL)        # learned position table (paper-transformer)
pos = pos_emb(torch.arange(S))              # (S, D_MODEL), one vector per slot
x = tok + pos                               # broadcast-add -> (1, S, D_MODEL)
print("token:", tuple(tok.shape), " + position:", tuple(pos.shape), " -> input x:", tuple(x.shape))

### Step 7 — Project the input into queries, keys, and values

Attention asks, for each position: *which other positions should I read from?* To decide, each token vector is projected three ways by learned matrices: a **query** (what I'm looking for), a **key** (what I offer), and a **value** (what I'll hand over). All three keep the `(1, S, D_MODEL)` shape.

In [ ]:
torch.manual_seed(0)
W_q = nn.Linear(D_MODEL, D_MODEL)
W_k = nn.Linear(D_MODEL, D_MODEL)
W_v = nn.Linear(D_MODEL, D_MODEL)

Q = W_q(x)
K = W_k(x)
V = W_v(x)
print("Q:", tuple(Q.shape), "  K:", tuple(K.shape), "  V:", tuple(V.shape))

**🎛️ Play with it — a dot product is a similarity score**

Every attention score is a dot product `query · key`, which is large when the two vectors point the same way and negative when they point opposite. **Try:** drag the query's angle. **Watch:** which key gets the highest bar. **Why it matters:** this is the entire mechanism by which a position decides *who to listen to*.

In [ ]:
key_dirs = {"east": 0, "north": 90, "west": 180, "south": 270}
keys = {k: np.array([np.cos(np.radians(a)), np.sin(np.radians(a))]) for k, a in key_dirs.items()}

def dot_demo(query_angle=30):
    q = np.array([np.cos(np.radians(query_angle)), np.sin(np.radians(query_angle))])
    sc = {k: float(q @ v) for k, v in keys.items()}
    fig, (axv, axb) = plt.subplots(1, 2, figsize=(9, 3.4))
    for k, v in keys.items():
        axv.arrow(0, 0, v[0], v[1], head_width=0.05, color="#888", length_includes_head=True)
        axv.text(v[0] * 1.18, v[1] * 1.18, k, ha="center")
    axv.arrow(0, 0, q[0], q[1], head_width=0.07, color="#ff6b6b", length_includes_head=True)
    axv.set_xlim(-1.4, 1.4); axv.set_ylim(-1.4, 1.4); axv.set_aspect("equal")
    axv.set_title("query (red) vs keys (grey)")
    axb.bar(list(sc), list(sc.values()), color="#4ea1ff"); axb.axhline(0, color="k", lw=0.5)
    axb.set_ylim(-1.1, 1.1); axb.set_title("dot-product score per key")
    plt.show()
    print("best match:", max(sc, key=sc.get))

try:
    from ipywidgets import interact, FloatSlider
    interact(dot_demo, query_angle=FloatSlider(value=30, min=0, max=360, step=5))
except Exception:
    dot_demo()

### Step 8 — Score every pair of positions

The relevance of position `j` to position `i` is the dot product of query `i` with key `j`, divided by `√D_MODEL` (the scaling keeps the numbers from blowing up as the width grows). That gives an `S × S` score matrix. **Note it is still full** — every position can currently see every other, including the future. We fix that next.

In [ ]:
scores = Q @ K.transpose(-2, -1) / math.sqrt(D_MODEL)   # (1, S, S)
print("scores:", tuple(scores.shape))

plt.figure(figsize=(5, 4.4))
plt.imshow(scores[0].detach(), cmap="magma")
plt.xticks(range(S), list(ctx)); plt.yticks(range(S), list(ctx))
plt.xlabel("key position j"); plt.ylabel("query position i")
plt.title("raw scores Q·Kᵀ/√d  (still un-masked)")
plt.colorbar(label="score"); plt.show()

**🎛️ Play with it — why divide by √d?**

Dot products of random `d`-dimensional vectors grow like `√d`, so without the `1/√d` scaling the scores get huge and softmax collapses onto a single key (which starves the gradient). **Try:** raise `d`. **Watch:** the un-scaled softmax (left, red) spikes toward 1.0 while the √d-scaled one (right, green) stays balanced. **Why it matters:** the scaling keeps attention trainable as the model gets wider.

In [ ]:
def scale_demo(d=64, n_keys=6):
    torch.manual_seed(0)
    q  = torch.randn(d)
    Ks = torch.randn(n_keys, d)
    raw      = Ks @ q
    unscaled = torch.softmax(raw, 0)
    scaled   = torch.softmax(raw / (d ** 0.5), 0)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
    a1.bar(range(n_keys), unscaled.tolist(), color="#ff6b6b"); a1.set_ylim(0, 1)
    a1.set_title(f"WITHOUT /\u221ad   (max={unscaled.max():.2f})")
    a2.bar(range(n_keys), scaled.tolist(), color="#7ee787")
    a2.set_title(f"WITH /\u221ad   (max={scaled.max():.2f})")
    for a in (a1, a2):
        a.set_xlabel("key index")
    plt.show()

try:
    from ipywidgets import interact, IntSlider
    interact(scale_demo, d=IntSlider(value=64, min=1, max=512, step=1),
             n_keys=IntSlider(value=6, min=2, max=12))
except Exception:
    scale_demo()

### Step 9 — Forbid the future with a causal mask

GPT predicts the next character from only the ones **before** it, so position `i` must not attend to any `j > i`. We set those upper-triangle scores to `-inf`; after the softmax they become exactly zero. The heatmap blanks out the forbidden future (the upper triangle).

In [ ]:
mask = torch.triu(torch.ones(S, S), diagonal=1).bool()   # True ABOVE the diagonal = the future
scores_masked = scores.masked_fill(mask, float("-inf"))

show = scores_masked[0].detach().clone()
show[mask] = float("nan")                                # blank the -inf cells for display
plt.figure(figsize=(5, 4.4))
plt.imshow(show, cmap="magma")
plt.xticks(range(S), list(ctx)); plt.yticks(range(S), list(ctx))
plt.xlabel("key position j"); plt.ylabel("query position i")
plt.title("masked scores (future blanked)")
plt.colorbar(label="score"); plt.show()

**🎛️ Play with it — the causal mask**

The mask is what makes this a *predict-the-next* model. **Try:** grow `S`, then untick **causal**. **Watch:** with causal off, every query can see every key (a bidirectional encoder like BERT); with it on, the upper triangle goes dark. **Why it matters:** next-character training would be trivially cheated if a position could peek at the answer to its right.

In [ ]:
def mask_demo(S=8, causal=True):
    m = torch.triu(torch.ones(S, S), 1).bool() if causal else torch.zeros(S, S, dtype=torch.bool)
    plt.figure(figsize=(4.6, 4))
    plt.imshow((~m).int(), cmap="Blues")
    plt.title("allowed positions — " + ("causal" if causal else "full (bidirectional)"))
    plt.xlabel("key j"); plt.ylabel("query i"); plt.colorbar(label="allowed (1)")
    plt.show()
    print("each query attends to",
          "only j <= i (past + present)" if causal else "ALL positions (past AND future)")

try:
    from ipywidgets import interact, IntSlider, Checkbox
    interact(mask_demo, S=IntSlider(value=8, min=2, max=20), causal=Checkbox(value=True))
except Exception:
    mask_demo()

### Step 10 — Softmax the scores into attention weights

A row-wise softmax turns each row of scores into a probability distribution over the allowed positions: the **attention weights**. Every row is lower-triangular and sums to 1 — position `i` spreads 100% of its attention across positions `0…i`.

In [ ]:
weights = F.softmax(scores_masked, dim=-1)               # (1, S, S), rows sum to 1
print("row sums:", [round(v, 3) for v in weights[0].sum(-1).tolist()])

plt.figure(figsize=(5, 4.4))
plt.imshow(weights[0].detach(), cmap="viridis")
plt.xticks(range(S), list(ctx)); plt.yticks(range(S), list(ctx))
plt.xlabel("attends to (key)"); plt.ylabel("query position")
plt.title("attention weights (rows sum to 1)")
plt.colorbar(label="weight"); plt.show()

**🎛️ Play with it — one query's attention, and temperature**

Now read attention from a single position's point of view, using the real scores from Step 8. **Try:** pick a query position `i`, then vary the temperature `T`. **Watch:** position `i` only ever attends to itself and earlier characters (causality); low `T` concentrates the weight on its favourite key, high `T` flattens it toward uniform. **Why it matters:** this same temperature knob will reappear at generation time to control how decisive the model is.

In [ ]:
ctx_chars = list(ctx)

def attn_play(i=12, T=1.0):
    row = scores[0, i].clone() / T        # raw scores for query i, temperature-scaled
    row[mask[i]] = float("-inf")          # re-apply causality (no peeking to the right)
    w = torch.softmax(row, 0)
    plt.figure(figsize=(8, 3))
    plt.bar(range(len(ctx_chars)), w.detach().tolist(), color="#79c0ff")
    plt.xticks(range(len(ctx_chars)), ctx_chars)
    plt.ylim(0, 1); plt.ylabel("weight")
    plt.title(f"query {i} ('{ctx_chars[i]}') attends over keys   (T={T})")
    plt.show()
    print("attends most to:", repr(ctx_chars[int(w.argmax())]), " weight =", round(float(w.max()), 3))

try:
    from ipywidgets import interact, IntSlider, FloatSlider
    interact(attn_play, i=IntSlider(value=S - 1, min=0, max=S - 1),
             T=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1))
except Exception:
    attn_play()

### Step 11 — Mix the values

Finally each position builds its output as the weighted sum of the **value** vectors, using its row of weights: `out = weights @ V`. Position `i` is now a blend of the value vectors of positions `0…i` — information has flowed from the past into the present. Output shape matches the input.

In [ ]:
out = weights @ V                                        # (1, S, D_MODEL)
print("weights:", tuple(weights.shape), " @ V:", tuple(V.shape), " -> out:", tuple(out.shape))
print("out matches input shape:", tuple(out.shape) == tuple(x.shape))

### Step 12 — Multiple heads, packaged into a reusable module

Real attention runs `h` of these score→mask→softmax→mix computations **in parallel** on different `D_MODEL/h`-sized slices (the "heads"), so different heads can specialise, then concatenates them and mixes with `W_o`. The class below is exactly Steps 7–11 wrapped up, generalised to `h` heads, so we can stack it.

In [ ]:
# Causal multi-head self-attention (paper-attention + paper-transformer + paper-gpt's mask).
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, h, max_len):
        super().__init__()
        assert d_model % h == 0
        self.h = h
        self.d_k = d_model // h
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        # paper-gpt's causal mask: True ABOVE the diagonal = the future, which we forbid.
        self.register_buffer("mask", torch.triu(torch.ones(max_len, max_len), diagonal=1).bool())

    def _split(self, x):                                    # (B,S,d_model) -> (B,h,S,d_k): the "heads"
        B, S, _ = x.shape
        return x.view(B, S, self.h, self.d_k).transpose(1, 2)

    def forward(self, x):
        B, S, _ = x.shape
        Q = self._split(self.W_q(x))                                   # Step 7, per head
        K = self._split(self.W_k(x))
        V = self._split(self.W_v(x))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)         # Step 8 (paper-attention)
        scores = scores.masked_fill(self.mask[:S, :S], float("-inf"))  # Step 9 (paper-gpt)
        weights = F.softmax(scores, dim=-1)                            # Step 10
        out = weights @ V                                              # Step 11
        out = out.transpose(1, 2).contiguous().view(B, S, self.h * self.d_k)  # concat the heads
        return self.W_o(out)

torch.manual_seed(0)
attn = CausalSelfAttention(D_MODEL, h=4, max_len=SEQ)
print("multi-head output:", tuple(attn(x).shape), " (same (1, S, D_MODEL) shape as the input)")

**🎛️ Play with it — how many heads?**

Multi-head attention splits the `D_MODEL = 64` channels into `h` groups and runs attention in each. **Try:** change `h` (it must divide 64). **Watch:** each head's width `d_k = 64/h` shrinks as `h` grows, while the output shape stays the same. **Why it matters:** more heads = more independent relationships the layer can track at once, each in a smaller subspace.

In [ ]:
def heads_demo(h=4):
    if D_MODEL % h:
        print(f"h={h} does not divide D_MODEL={D_MODEL} — pick a divisor.")
        return
    torch.manual_seed(0)
    layer = CausalSelfAttention(D_MODEL, h=h, max_len=SEQ)
    out = layer(x)
    print(f"h = {h:2d}   ->   d_k = {D_MODEL // h:2d} channels per head,   output {tuple(out.shape)}")

try:
    from ipywidgets import interact, Dropdown
    interact(heads_demo, h=Dropdown(options=[1, 2, 4, 8, 16, 32, 64], value=4))
except Exception:
    heads_demo()

### Step 13 — The pre-norm transformer block

A `Block` wraps two sublayers: the causal attention from Step 12 and a small feed-forward MLP. Each sublayer is *pre-normed* — a `LayerNorm` is applied first (the LayerNorm paper). The two `x + ...` lines are ResNet's residual connections: the block adds its output back onto the input, which keeps gradients flowing through deep stacks. Because of those adds, the output shape equals the input shape — the "residual stream" — which we confirm below.

In [ ]:
# Pre-norm block: LayerNorm -> sublayer -> residual add (paper-layernorm + paper-resnet).
class Block(nn.Module):
    def __init__(self, d_model, h, max_len, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)                    # pre-norm: LN at the input (paper-layernorm)
        self.attn = CausalSelfAttention(d_model, h, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # residual around masked attention (paper-resnet)
        x = x + self.ff(self.ln2(x))       # residual around feed-forward    (paper-resnet)
        return x

demo_block = Block(D_MODEL, h=4, max_len=SEQ, d_ff=128)
print("block in :", tuple(x.shape))
print("block out:", tuple(demo_block(x).shape), " (unchanged — the residual stream)")

### Step 14 — Assemble the nanoGPT

The full model owns its **token + positional embeddings** (Steps 5–6), stacks `n_layers` blocks, applies a final `LayerNorm`, and projects to per-character logits with the language-model `head`. `generate` runs it autoregressively: take the logits at the last position, divide by a temperature, softmax, and *sample* the next character (GPT's sampling loop).

In [ ]:
# The nanoGPT: token + learned positional embeddings, N blocks, final LN, vocab head.
class NanoGPT(nn.Module):
    def __init__(self, vocab, d_model=64, h=4, n_layers=3, max_len=64, d_ff=128):
        super().__init__()
        self.max_len = max_len                              # longest context the position table can index
        self.tok = nn.Embedding(vocab, d_model)             # token embedding table (paper-word2vec)
        self.pos = nn.Embedding(max_len, d_model)           # LEARNED positional embedding (paper-transformer)
        self.blocks = nn.ModuleList([Block(d_model, h, max_len, d_ff) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)                   # GPT-2's extra final LayerNorm
        self.head = nn.Linear(d_model, vocab)               # the language-model head -> per-char logits

    def forward(self, idx):
        B, S = idx.shape
        pos = torch.arange(S, device=idx.device)
        x = self.tok(idx) + self.pos(pos)                   # add token + positional embeddings
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x))                      # (B,S,vocab) next-char logits

    @torch.no_grad()
    def generate(self, idx, n_new, temp=0.8):
        for _ in range(n_new):
            logits = self(idx[:, -self.max_len:])[:, -1, :] / temp   # logits for the next char at the last slot
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1)               # SAMPLE the next char (paper-gpt)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

### Step 15 — Trace the shapes through the model

Before training, push our context through an untrained model and record the shape at every stage. This is the data flow in one table: ids → embeddings → blocks (unchanged residual stream) → logits over the vocabulary at every position.

In [ ]:
torch.manual_seed(0)
net = NanoGPT(VOCAB, max_len=SEQ)

with torch.no_grad():
    stages = [("input ids", tuple(ids.shape))]
    h = net.tok(ids) + net.pos(torch.arange(S))
    stages.append(("+ embeddings", tuple(h.shape)))
    for bi, blk in enumerate(net.blocks):
        h = blk(h)
        stages.append((f"after block {bi}", tuple(h.shape)))
    logits = net.head(net.ln_f(h))
    stages.append(("logits (per-char)", tuple(logits.shape)))

display(pd.DataFrame(stages, columns=["stage", "shape"]))

### Step 16 — One generation step (before any training)

`generate` adds one character at a time. Here is a single step on the **untrained** model: read the logits at the last position, softmax to probabilities, and sample. Untrained, the pick is essentially random — that is exactly what training will fix.

In [ ]:
with torch.no_grad():
    last_logits = net(ids)[0, -1]                # logits for the char after the context
    probs = F.softmax(last_logits, dim=-1)
    nxt = torch.multinomial(probs, 1).item()
print("context     :", repr(ctx))
print("sampled next:", repr(itos[nxt]), " (untrained -> basically random)")
print("max prob    :", round(probs.max().item(), 3), " (near 1/VOCAB =", round(1 / VOCAB, 3), "when clueless)")

### Step 17 — Count the parameters and check the random-init loss

Two sanity checks. First, tabulate the trainable parameters by component — most of the capacity lives in the transformer blocks, not the embeddings. Second, run one **untrained** batch: its cross-entropy should land near `ln(VOCAB)`, because a clueless model spreads its bets evenly over `VOCAB` characters.

In [ ]:
import re
group_counts = {}
for name, p in net.named_parameters():
    m = re.match(r"(blocks\.\d+)", name)
    comp = m.group(1) if m else name.split(".")[0]
    group_counts[comp] = group_counts.get(comp, 0) + p.numel()

total = sum(group_counts.values())
param_df = pd.DataFrame({
    "component":  list(group_counts),
    "parameters": list(group_counts.values()),
    "% of total": [round(100 * v / total, 1) for v in group_counts.values()],
})
display(param_df)
print("total trainable parameters:", f"{total:,}")

with torch.no_grad():
    xb, yb = get_batch()
    loss0 = F.cross_entropy(net(xb).reshape(-1, VOCAB), yb.reshape(-1)).item()
print(f"untrained loss = {loss0:.3f}   vs   ln(VOCAB) = {float(np.log(VOCAB)):.3f}   (should match)")

### Step 18 — What one training step does

Before the full loop, watch a single optimiser step. We forward one batch, read the loss, backprop, and step **AdamW** (Adam with decoupled weight decay). Re-measuring the loss on the *same* batch shows it drop — one nudge downhill.

In [ ]:
opt = torch.optim.AdamW(net.parameters(), lr=3e-3)        # paper-adamw
xb, yb = get_batch()
before = F.cross_entropy(net(xb).reshape(-1, VOCAB), yb.reshape(-1))
opt.zero_grad(); before.backward(); opt.step()            # one gradient step
after = F.cross_entropy(net(xb).reshape(-1, VOCAB), yb.reshape(-1))
print(f"loss on this batch: before = {before.item():.3f}  ->  after one step = {after.item():.3f}")

### Step 19 — The full training loop (recording the loss curve)

Now the real run. We start from a **fresh** model so the recorded curve begins at initialisation, then optimise with AdamW on the next-character cross-entropy — GPT's causal-LM objective. We record the loss at every step and a generated sample every 500 steps. The loss should fall from ~3.0 toward ~0.2 and the samples should drift from gibberish to readable text. (Our small run; exact values vary by hardware and seed.)

In [ ]:
def sample(net, n_new=60, seed_char="t"):
    start = torch.tensor([[stoi[seed_char]]])
    out = net.generate(start, n_new)[0].tolist()
    return "".join(itos[i] for i in out)

torch.manual_seed(0)
net = NanoGPT(VOCAB, max_len=SEQ)                          # fresh model for a clean curve
opt = torch.optim.AdamW(net.parameters(), lr=3e-3)
loss_hist, sample_log = [], []
for step in range(2001):
    x, y = get_batch()
    loss = F.cross_entropy(net(x).reshape(-1, VOCAB), y.reshape(-1))  # causal-LM objective (paper-gpt)
    opt.zero_grad()
    loss.backward()
    opt.step()
    loss_hist.append((step, loss.item()))
    if step % 500 == 0:
        s = sample(net)
        sample_log.append((step, s))
        print(f"step {step:4d}  loss {loss.item():.3f}   sample: {s!r}")
# Loss falls from ~3.0 (random over ~20 chars) toward ~0.2, and samples go gibberish -> readable.
# Our small run, not a paper's reported number; exact values vary by hardware and seed.

## Visualize the data & results

_As the assembled char-level mini-GPT trains with AdamW on the next-character cross-entropy objective, does the loss fall and does the generated text become readable?_ We answer with the history we just recorded — no retraining needed.

### Step 20 — The loss curve

Plotting every recorded step shows the full optimisation trajectory. The dashed line marks `ln(VOCAB)`, the loss of a model that has learned nothing; a working build starts there and drops well below it.

In [ ]:
steps  = [s for s, _ in loss_hist]
losses = [l for _, l in loss_hist]

plt.figure(figsize=(7.5, 3.6))
plt.plot(steps, losses, color="#7ee787", lw=1.2, label="train loss")
plt.axhline(float(np.log(VOCAB)), color="#ff6b6b", ls="--",
            label=f"ln(VOCAB) = {float(np.log(VOCAB)):.2f}  (random guessing)")
plt.xlabel("training step"); plt.ylabel("next-char cross-entropy (nats)")
plt.title("loss falls from ~ln(VOCAB) toward ~0")
plt.legend(); plt.show()

print("first-step loss:", round(losses[0], 3), "   final-step loss:", round(losses[-1], 3))

### Step 21 — Samples: gibberish → readable

The same story in words. Each row is a 60-character sample (seeded with `'t'`) taken at that training step. Early rows are noise; later rows reproduce fragments of the corpus.

In [ ]:
sample_df = pd.DataFrame(sample_log, columns=["step", "generated sample (60 chars, seeded with 't')"])
display(sample_df)

**🎛️ Play with it — temperature when generating**

Generation samples one character at a time from the softmax over next-char logits, divided by a temperature. **Try:** sweep `T` from 0.2 to 1.5 and change the seed character. **Watch:** low `T` is safe and repetitive (it almost always picks the most likely char); high `T` is adventurous and starts making spelling mistakes. **Why it matters:** temperature is the single most common knob for trading off coherence vs creativity in real LLMs.

In [ ]:
def gen_demo(T=0.8, seed="t"):
    if seed not in stoi:
        print("pick a character that is in the vocabulary"); return
    torch.manual_seed(0)
    start = torch.tensor([[stoi[seed]]])
    out = net.generate(start, 80, temp=T)[0].tolist()
    print(f"T={T}:  {''.join(itos[i] for i in out)!r}")

try:
    from ipywidgets import interact, FloatSlider, Dropdown
    interact(gen_demo, T=FloatSlider(value=0.8, min=0.2, max=1.5, step=0.1),
             seed=Dropdown(options=[c for c in chars if c.isalpha()], value="t"))
except Exception:
    gen_demo()

### Step 22 — What the trained model predicts next

A language model outputs a probability distribution over the next character. We feed it `"to be, or not to b"` and bar-chart its top predictions — it should put almost all of its mass on `'e'` (…to b → e).

In [ ]:
context = "to be, or not to b"
cids = torch.tensor([encode(context)])
with torch.no_grad():
    logits = net(cids)[:, -1, :]           # logits at the last position
    probs = F.softmax(logits, dim=-1)[0]

top = torch.argsort(probs, descending=True)[:10]
plt.figure(figsize=(7, 3))
plt.bar([repr(itos[i.item()]) for i in top], probs[top].tolist(), color="#79c0ff")
plt.title(f"P(next character | {context!r})")
plt.ylabel("probability")
plt.show()

print("context           :", repr(context))
print("top next character:", repr(itos[top[0].item()]), " with p =", round(probs[top[0]].item(), 3))

### Step 23 — The trained model's attention

Finally, the same tensor walk from Steps 7–10, but on the **trained** model (block 0, head 0). The lower-triangular shape is the causal mask from Step 9; the bright cells show which earlier characters each position now leans on after training.

In [ ]:
context = "to be, or not to b"
cids = torch.tensor([encode(context)])
Sc = cids.shape[1]
a0 = net.blocks[0].attn
with torch.no_grad():
    x_emb = net.tok(cids) + net.pos(torch.arange(Sc))
    x_ln = net.blocks[0].ln1(x_emb)                      # pre-norm input the attention actually sees
    Q = a0._split(a0.W_q(x_ln))
    K = a0._split(a0.W_k(x_ln))
    sc = Q @ K.transpose(-2, -1) / math.sqrt(a0.d_k)
    sc = sc.masked_fill(a0.mask[:Sc, :Sc], float("-inf"))
    w = torch.softmax(sc, dim=-1)[0, 0]                  # head 0

fig, ax = plt.subplots(figsize=(6, 5.2))
im = ax.imshow(w, cmap="viridis")
ax.set_xticks(range(Sc)); ax.set_xticklabels(list(context))
ax.set_yticks(range(Sc)); ax.set_yticklabels(list(context))
ax.set_xlabel("attends to (key)"); ax.set_ylabel("query position")
ax.set_title("trained attention — block 0, head 0")
fig.colorbar(im, label="weight")
plt.show()

**🎛️ Play with it — what the embeddings learned**

The token-embedding table started as random vectors; training nudged characters that play similar roles closer together. **Try:** pick a character and read its nearest neighbours by cosine similarity. **Watch:** vowels, or characters that often appear in the same words, tend to cluster. **Why it matters:** this is the same idea as word2vec — *meaning becomes geometry* — here emerging for free from next-character prediction.

In [ ]:
def neighbours(c="e", k=6):
    if c not in stoi:
        print("pick a vocab character"); return
    E  = net.tok.weight.detach()
    En = E / E.norm(dim=1, keepdim=True)
    sims = En @ En[stoi[c]]
    order = [j for j in sims.argsort(descending=True).tolist() if j != stoi[c]][:k]
    plt.figure(figsize=(7, 3))
    plt.bar([repr(itos[j]) for j in order], [float(sims[j]) for j in order], color="#c89bff")
    plt.ylim(0, 1); plt.ylabel("cosine similarity")
    plt.title(f"nearest neighbours of {c!r} in the trained embedding space")
    plt.show()

try:
    from ipywidgets import interact, Dropdown, IntSlider
    interact(neighbours, c=Dropdown(options=list(chars), value="e"),
             k=IntSlider(value=6, min=2, max=12))
except Exception:
    neighbours()